# Game 8 - The Cross-Modal Truth Arena

**Team:** Ded_Sec

This completed notebook documents the final multimodal system and loads the
generated validation and public-test artifacts. The image expert is the
optimized 160-pixel scratch CNN from Game 6. The text expert is the local
MiniLM encoder plus logistic classifier and quality features from Game 6.
A locally shipped CLIP encoder adds a semantic image-text agreement score
(clip_similarity) so mismatched pairs are detectable even when neither
component appears in the canonical manifests.

The relation layer measures canonical image-text pairing, whether both
components are known, model agreement, CLIP similarity, and the probability
gap between the two modalities. It compares image-only, text-only, weighted
fusion, relation-gated fusion, logistic stacking, gradient-boosted stacking,
and the average of the two stacking models. The stacking models are fit on
the original rows plus two relation-blind copies (one clean, one with CLIP
jitter) so they generalize to hidden tests built from entirely new material.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
comparison = pd.read_csv(ROOT / "game8_model_comparison.csv")
evidence = pd.read_csv(ROOT / "game8_evidence_analysis.csv")
public = pd.read_csv(ROOT / "game8_public_predictions.csv")
comparison.sort_values("macro_f1", ascending=False)

## Selected system

`relation_gradient_boosting` was selected by validation macro-F1.

- Supplied benchmark accuracy: 0.9340
- Supplied benchmark macro-F1: 0.9340
- Image weight in the simple fusion baseline: 0.500

Canonical mismatches are strong evidence for `fake`. For matched pairs, the
fusion model decides how much to trust the image and text experts from their
probabilities, confidence, agreement, and interaction features.

This benchmark score uses all canonical manifests available from earlier games.
It is a competition benchmark result, not a guaranteed accuracy on unrelated
real-world data.


In [ ]:
pd.crosstab(
    evidence["primary_evidence"],
    [evidence["correct"], evidence["reviewer_flag"]],
    margins=True,
)

In [ ]:
errors = evidence.loc[~evidence["correct"]]
errors[[
    "sample_id", "true_label", "predicted_label", "confidence",
    "primary_evidence", "reviewer_flag", "error_category"
]].head(30)

In [ ]:
required = [
    "sample_id", "predicted_label", "confidence",
    "primary_evidence", "reviewer_flag",
]
assert public.columns.tolist() == required
assert public["primary_evidence"].isin(
    ["image", "text", "image_text_relation", "uncertain"]
).all()
assert public["confidence"].between(0, 1).all()
public.head(10)

## Trust and review policy

Cases are flagged for review when confidence is below 0.68, when the image and
text experts strongly disagree, or when a detected relation mismatch is not
supported with at least 0.90 confidence. The evidence field is intentionally
limited to the four values required by the release contract.
